In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error  
from sklearn.datasets import fetch_california_housing


from urllib.parse import urlparse


from mlflow.models.signature import infer_signature



In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")


In [ ]:
housing = fetch_california_housing()
housing.keys()


In [ ]:
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = pd.DataFrame(housing.target, columns=["MedHouseVal"])


In [ ]:
X.shape, y.shape


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:
signature = infer_signature(X_train, y_train)


In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}



In [ ]:
def hyperparam_tuning(param_grid, X_train, y_train, model = RandomForestRegressor()):
    grid_search = GridSearchCV(
        estimator=model, 
        param_grid=param_grid, 
        cv=3, 
        scoring="neg_mean_squared_error",
        n_jobs=-1,
        verbose=2
    )
    grid_search.fit(X_train, y_train)
    return grid_search.best_estimator_


In [20]:
model = RandomForestRegressor()
model_name = "Housing-RandomForestRegressor"
mlflow.set_experiment("housing-regression")


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1769946346750, experiment_id='1', last_update_time=1769946346750, lifecycle_stage='active', name='housing-regression', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [ ]:
mlflow.sklearn.autolog(
    log_models=False,
    silent=True
)


In [21]:


with mlflow.start_run(run_name="hparam_tuning"):
    best_model = hyperparam_tuning(param_grid, X_train, y_train)

    y_pred = best_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = float(np.sqrt(mse))

    # best_params = {
    #     f"best_{k}": v
    #     for k, v in best_model.get_params().items()
    #     if k in param_grid
    # }
    # mlflow.log_params(best_params)
    
    mlflow.log_metrics({"testing_mean_squared_error": mse, "testing_root_mean_squared_error": rmse})

    signature = infer_signature(X_train, best_model.predict(X_train))

    mlflow.sklearn.log_model( 
                             sk_model=best_model, 
                             name=model_name, 
                             signature=signature,
                            #  registered_model_name=model_name, 
                             input_example=X_train[:5],
                             )
    



Fitting 3 folds for each of 24 candidates, totalling 72 fits


c:\Users\mathe\repos\study\UDEMY\mlops\7\.venv\lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


🏃 View run bustling-toad-462 at: http://localhost:5000/#/experiments/1/runs/64ed2a220f384ed8b5610203e3184f64
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run overjoyed-hound-733 at: http://localhost:5000/#/experiments/1/runs/8e8a6236612f46528e93d1982c3754b2
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run righteous-goose-84 at: http://localhost:5000/#/experiments/1/runs/7d289954919c4729b388d76ce1d1aa36
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run dazzling-owl-371 at: http://localhost:5000/#/experiments/1/runs/7808dd9c2faf447c9c4e8d343e131ab0
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run orderly-finch-389 at: http://localhost:5000/#/experiments/1/runs/17f019cd03fd468cbd08142916238b8b
🧪 View experiment at: http://localhost:5000/#/experiments/1


c:\Users\mathe\repos\study\UDEMY\mlops\7\.venv\lib\site-packages\mlflow\models\model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)
2026/02/01 09:48:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run hparam_tuning at: http://localhost:5000/#/experiments/1/runs/7910c03155e04d49b4d0f83d0da4339b
🧪 View experiment at: http://localhost:5000/#/experiments/1


![](pics/image.png)
